# Climate Model Skill Metric Plots

This notebook walks through how to generate heatmap plots for Potential Skill, Unconditional Bias, Conditional Bias, and Skill Score for a precipitation climate model assuming that you have monthly aligned precipitation data in netCDF format pre-processed by 'monthly_merged_data_generation.py'

# Helper Functions

The code cell below defines helper functions for loading in the data and pre-processing it for plotting.

In [ ]:
# import necessary packages
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import os

# set global seaborn plotting options
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

# define some helper functions for computations and loading

def load_data(df_dict, region_list, model_list, monthly_netcdf_path):
    """
    Loads data from netCDF files into a dictionary of pandas DataFrames.

    Parameters:
    region_list (list): List of region names.
    model_list (list): List of model names.
    monthly_netcdf_path (str): Path to a directory containing monthly
                                netCDF files. (i.e data/netCDF/monthly)
    df_dict (dict): Dictionary to store loaded DataFrames. Keys are tuples
                    of (region, model) and values are pandas DataFrames.

    Assumptions:
    Assumes that you have a directory containing netCDF files named as
    'region_model_merged.nc', and that those files have been generated
    by monthly_data_generation.py.
    """
    # Normalize the path to prevent double slashes
    monthly_netcdf_path = os.path.normpath(monthly_netcdf_path)

    # loop through all files
    for region in region_list:
        for model in model_list:

            # get the file name from netcdf folder
            filename = os.path.join(monthly_netcdf_path, f'{region}_{model}_merged.nc')
            try:
                # open the netcdf file, convert to dataframe, reset index
                # store into dataframes dictionary with key as {region, model)
                df_dict[(region, model)] = xr.open_dataset(filename).to_dataframe().reset_index()

            # simple error handling
            except FileNotFoundError:
                print(f"Missing file: {filename}")
            except Exception as e:
                print(f"Error loading {filename}: {e}")


def compute_statistics(dataframes_dict, start_year, end_year):
    """Computes relevant statistics for each region and model combination.

    Parameters:
    dataframes_dict (dict): A dictionary where keys are tuples of (region, model)
                            and values are pandas DataFrames.
                            Generated by load_data function.
    start_year (int): Start year for subsetting data.
    end_year (int): End year for subsetting data.

    Modifies dataframes_dict:
          A dictionary where keys are tuples of (region, model)
          and values are pandas DataFrames containing computed statistics.
          Statistics include:
            - corr: Spearman correlation coefficient
            - potential_skill: Squared correlation coefficient
            - conditional_bias: Squared difference between correlation
                                and ratio of predicted to actual standard deviation
            - unconditional_bias: Squared difference between predicted
                                and actual mean divided by actual standard deviation

    Columns for a stat dataframe of one (region, model) combination:
            ['month', 'lead_time', 'pred_mean', 'pred_std', 'actual_mean',
            'actual_std', 'corr', 'potential_skill', 'conditional_bias',
            'unconditional_bias', 'skill_score']
    """
    # iterate over all (region, model) and dataframe pairs in the dataframes_dict
    for key, df in dataframes_dict.items():

        # drop all na values (i.e. ocean areas)
        df = df.dropna()

        # subset data to years of interest
        df = df[(df['time'].dt.year >= start_year) & (df['time'].dt.year <= end_year)]

        # compute the ensemble mean
        ens_mean = df.groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']].mean().reset_index()

        # compute the spatial mean
        spatial = ens_mean.groupby(['time', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()

        # convert precip to mm/day from mm/month
        spatial['precip'] /= 30

        # create month column for plotting
        spatial['month'] = spatial['time'].dt.month

        # compute spearman correlation
        corr = (spatial.groupby(['month', 'lead_time'])[['predicted_precip', 'precip']]
                      .corr(method='spearman')
                      .iloc[0::2, -1]
                      .droplevel(-1)
                      .reset_index()
                      .rename(columns={'precip': 'corr'}))

        # create stat dataframe
        # spatial means with mean and standard deviation computed by unique
        # month and lead time combinations
        stat = (spatial.drop('time', axis=1)
                      .groupby(['month', 'lead_time'])
                      .agg(['mean', 'std'])
                      .reset_index())
        stat.columns = ['month', 'lead_time', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

        # merge correlation with stat dataframe on month and lead time
        all_stat = stat.merge(corr, on=['month', 'lead_time'], how='left')

        # compute other relevant statistics
        all_stat['potential_skill'] = np.square(all_stat['corr'])
        all_stat['conditional_bias'] = np.square(all_stat['corr'] - (all_stat['pred_std'] / all_stat['actual_std']))
        all_stat['unconditional_bias'] = np.square((all_stat['pred_mean'] - all_stat['actual_mean']) / all_stat['actual_std'])
        all_stat['skill_score'] = all_stat['potential_skill'] - all_stat['conditional_bias'] - all_stat['unconditional_bias']

        # add current all_stat dataframe to dataframes dict in-place
        dataframes_dict[key] = all_stat

def apply_dry_mask(df):
    """
    Applies a mask to specified months for certain regions, setting specified columns to NaN.

    Parameters:
    df (DataFrame): The input DataFrame containing 'region' and 'month' columns.

    Returns:
    mask_df: The DataFrame with specified columns set to NaN for specified months and regions.
    """
    mask_df = df.copy()
    mask = mask_df.apply(lambda row: row['month'] in dry_mask_months.get(row['region'], []), axis=1)
    cols_to_nan = ['corr', 'potential_skill', 'conditional_bias', 'unconditional_bias', 'skill_score']
    mask_df.loc[mask, cols_to_nan] = np.nan
    return mask_df

# Data Pre-processing (for powerful computers)

The code below will load and pre-process the monthly data for all models and regions that it automatically finds, given the file path to the monthly data.

In [ ]:
'''

"""For computers with sufficient RAM, uncomment and run this portion for convenient automation.

Notes:
- Automatically generates region_list and model_list based on file names in monthly netCDF directory.
- Assumes that you have a directory containing netCDF files named as
'region_model_merged.nc', and that those files have been generated
by monthly_data_generation.py.

Usage Outline:
1. Modify monthly_netcdf_path to point to your monthly netCDF directory.
2. Run the script.
3. The script will automatically generate region_list and model_list based on file names in monthly netCDF directory.
   For example, if you have files 'west_africa_CanESM5_merged.nc' and 'west_africa_ECMWF_merged.nc' in your monthly netCDF directory,
   region_list will contain ['west_africa'] and model_list will contain ['CanESM5', 'ECMWF']
4. The script will load data, compute statistics, and apply dry masking.
   It gives you two DataFrames: stat_clean and stat_clean_dry_masked. These will be used to plot the heatmaps.
"""
# modify to your monthly netcdf path
monthly_netcdf_path = '/content/drive/My Drive/capstone_data/netCDF/'

# get list of files from monthly netCDF directory
files = os.listdir(monthly_netcdf_path)

# define empty region list
region_list = []

# define empty model list
model_list = []

# populate region list and model list automatically based on file names
for file_name in files:

    # extract region and model name from file name
    region_name = '_'.join(file_name.split('_')[0:-2])
    model_name = file_name.split('_')[-2]

    # append to respective list
    if region_name not in region_list:
        region_list.append(region_name)

    if model_name not in model_list:
        model_list.append(model_name)

# order lists alphabetically
region_list.order()
model_list.order()

# define months to dry mask by region, chosen from dry_masking.py
dry_mask_months = {
    'south_sudan': [1, 2, 12],
    'eastern_east_africa': [1, 2],
    'southern_africa': [5, 6, 7, 8, 9],
    'west_africa': [1, 2, 3, 11, 12]
}

# define start year and end year of interest for subsetting
start_year, end_year = 1993, 2024

# Initialize a dictionary to store all loaded data.
# This is initialized outside of the functions for memory reasons
dataframes_dict = {}

# load data into dataframes dict
load_data(dataframes_dict, region_list, model_list, monthly_netcdf_path)

# compute stats for all dataframes in dataframes dict
compute_statistics(dataframes_dict, start_year, end_year)

# Combine all stat dataframes into one DataFrame, apply dry masking
stat_clean = pd.concat(dataframes_dict.values(), keys=dataframes_dict.keys(), names=['region', 'model'])
stat_clean = stat_clean.reset_index().drop('level_2', axis=1)
stat_clean_dry_masked = apply_dry_mask(stat_clean)


'''

# Data pre-processing (for not-so-powerful computers)

The code below will load ad pre-process the monthly data for all models and regions that are manually defined in the lists, given a file path to the monthly data.

In [ ]:
"""For computers with insufficient RAM, run this portion. This defines specific
regions and models to load and perform calculations on.
In our case, for example, JMA is left out due to RAM constraints.

Notes:
- Specifically defined region_list and model_list based on availability of RAM.
- Assumes that you have a directory containing netCDF files named as
'region_model_merged.nc', and that those files have been generated
by monthly_data_generation.py.

Usage Outline:
1. Modify monthly_netcdf_path to point to your monthly netCDF directory.
2. Modify region_list and model_list to specify the regions and models you want to analyze.
   Ensure that the corresponding netCDF files exist in the specified directory.
   For example, if you want to analyze 'west_africa' region with 'CanESM5' and 'ECMWF' models,
   you would set region_list=['west_africa'] and model_list=['CanESM5', 'ECMWF'], then
   ensure that you have files 'west_africa_CanESM5_merged.nc' and 'west_africa_ECMWF_merged.nc'
   in your monthly netCDF directory.
3. Run the script.
4. The script will load data, compute statistics, and apply dry masking.
   It gives you two DataFrames: stat_clean and stat_clean_dry_masked. These will be used to plot the heatmaps.
"""
# modify to your monthly netcdf path
monthly_netcdf_path = 'data/netCDF/'

region_list = ['west_africa', 'southern_africa', 'eastern_ukraine',
               'eastern_east_africa', 'south_sudan', 'lake_victoria_basin',
               'sri_lanka']

model_list = ['CanESM5', 'ECMWF', 'DWD',
              'METEO', 'GFDL', 'GEM5', 'CMCC',
              'CCSM4', 'CESM1', 'NASA', 'NCEP']

unused_models = ['JMA']

# order lists alphabetically
region_list.order()
model_list.order()

# define months to dry mask by region, chosen from dry_masking.py
dry_mask_months = {
    'south_sudan': [1, 2, 12],
    'eastern_east_africa': [1, 2],
    'southern_africa': [5, 6, 7, 8, 9],
    'west_africa': [1, 2, 3, 11, 12]
}

# define start year and end year of interest for subsetting
start_year, end_year = 1993, 2024

# Initialize a dictionary to store all loaded data.
# This is initialized outside of the functions for memory reasons
dataframes_dict = {}

# load data into dataframes dict
load_data(dataframes_dict, region_list, model_list, monthly_netcdf_path)

# compute stats for all dataframes in dataframes dict
compute_statistics(dataframes_dict, start_year, end_year)

# Combine all stat dataframes into one DataFrame, apply dry masking
stat_clean = pd.concat(dataframes_dict.values(), keys=dataframes_dict.keys(), names=['region', 'model'])
stat_clean = stat_clean.reset_index().drop('level_2', axis=1)
stat_clean_dry_masked = apply_dry_mask(stat_clean)

Now that the data have been transformed to a suitible format for plotting, we can move on to the code below, which will draw and save the heatmap plots. Ensure that the save path for the images is changed to your save path for each plot code cell.

# Potential Skill Plots

In [ ]:
# helper function for potential skill heatmap
def draw_heatmap(*args, **kwargs):
    """Draws a heatmap for potential skill using the calculated metrics

       Note:
       Values bounded from 0 to 0.6, using Red colors for plotting
    """
    # access the dataframe that is passed into the function
    data = kwargs.pop('data')

    # pivot data, index is month, columns is lead time, values is potential skill
    d = data.pivot(index=args[1], columns=args[0], values=args[2])

    # define a heatmap with pivoted data, value bounds, and colors
    sns.heatmap(d, **kwargs, vmin=0, vmax=0.6,
                cmap=sns.color_palette('Reds', 12),
                linewidths=0.1, linecolor='black')

    # heatmap labels
    plt.xticks(np.arange(0.5, 12.5, 1))  # Set lead time ticks explicitly
    plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set lead time labels
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().invert_yaxis() # invert y axis; 1 appears at the bottom


"""Plot unmasked potential skill
"""
# Set large facetGrid, where columns is region and rows are models
fg = sns.FacetGrid(stat_clean, col='region', row='model', sharex=False, sharey=False)

# call helper function to draw heatmap onto the facet grid
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'potential_skill', square = True)

# set main and axis titles
fg.set_titles('Potential Skill \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")

# save figure, change to your save path
fg.savefig('figures/potential_skill.png')

# close for efficiency reasons
plt.close()

"""Plot masked potential skill
"""
# Each column is a different region, and each row is a different model
fg = sns.FacetGrid(stat_clean_dry_masked, col='region', row='model', sharex=False, sharey=False)

# call helper function to draw heatmap
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'potential_skill', square = True)

# set main and axis titles
fg.set_titles('Dry Masked Potential Skill \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")

# save figure, change to your save path
fg.savefig('figures/Model_Monthly_Metrics/Fig3_Monthly/dry_masked_potential_skill.png')

# close for efficiency reasons
plt.close()

# Unconditional Bias Plots

In [ ]:
def draw_heatmap(*args, **kwargs):
    """Draws a heatmap for unconditional bias using the calculated metrics

       Note:
       Values unbounded, using blue colors for plotting
    """
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, # add vmin=value,vmax=value, for bounding
                cmap=sns.color_palette('Blues', 10),
                linewidths=0.1, linecolor='black')
    plt.xticks(np.arange(0.5, 12.5, 1))  # Set xticks explicitly
    plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set xticklabels
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().invert_yaxis()

"""Plot unmasked unconditional bias
"""
fg = sns.FacetGrid(stat_clean, col='region', row='model', sharey=False, sharex=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'unconditional_bias', square = True)

fg.set_titles('Unconditional Bias \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/unconditional_bias.png') # save figure, change to your save path
plt.close()

"""Plot dry masked unconditional bias
"""
fg = sns.FacetGrid(stat_clean_dry_masked, col='region', row='model', sharey=False, sharex=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'unconditional_bias', square = True)

fg.set_titles('Dry Masked Unconditional Bias \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/Model_Monthly_Metrics/Fig3_Monthly/dry_masked_unconditional_bias.png') # save figure, change to your save path
plt.close()

# Conditional Bias Plots

In [ ]:
def draw_heatmap(*args, **kwargs):
    """Draws a heatmap for conditional bias using the calculated metrics

       Note:
       Values unbounded, using blue colors for plotting
    """
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, # add vmin=value,vmax=value, for bounding
                cmap=sns.color_palette('Blues', 10),
                linewidths=0.1, linecolor='black')
    plt.xticks(np.arange(0.5, 12.5, 1))  # Set xticks explicitly
    plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set xticklabels
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_yaxis()

"""Plot unasked conditional bias
"""
fg = sns.FacetGrid(stat_clean, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'conditional_bias', square = True)

fg.set_titles('Conditional Bias \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/conditional_bias.png') # save figure, change to your save path
plt.close()

"""Plot dry masked conditional bias
"""
fg = sns.FacetGrid(stat_clean_dry_masked, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'conditional_bias', square = True)

fg.set_titles('Dry Masked Conditional Bias \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/Model_Monthly_Metrics/Fig3_Monthly/dry_masked_conditional_bias.png') # save figure, change to your save path
plt.close()

# Skill Score Plots

In [ ]:
# Skill Score heatmap helper function
def draw_heatmap(*args, **kwargs):
    """Draws a heatmap for skill score using the calculated metrics

       Note:
       Values unbounded, using Blue colors for plotting
    """
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, # add vmin=value,vmax=value, for bounding
                cmap=sns.color_palette('vlag', 19),
                linewidths=0.1, linecolor='black')
    plt.xticks(np.arange(0.5, 12.5, 1))  # Set xticks explicitly
    plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set xticklabels
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().invert_yaxis()


"""Plot unmasked skill score
"""
fg = sns.FacetGrid(stat_clean, col='region', row='model',sharey=False, sharex=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'skill_score', square = True)


fg.set_titles('Skill Score \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/skill_score.png') # save figure, change to your save path
plt.close()


"""Plot dry masked skill score
"""
fg = sns.FacetGrid(stat_clean_dry_masked, col='region', row='model',sharey=False, sharex=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'skill_score', square = True)

fg.set_titles('Dry Masked Skill Score \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/Model_Monthly_Metrics/Fig3_Monthly/dry_masked_skill_score.png') # save figure, change to your save path
plt.close()